In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm import tqdm
import datasets
import collections

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
# Get the CACHE_ROOT path from the environment variable you just set in Linux
CACHE_ROOT = os.environ.get("HF_HOME")

if CACHE_ROOT is None:
    # This should not happen, but a safe fallback in case the variable was lost
    CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
    os.environ["HF_HOME"] = CACHE_ROOT

our_dataset = datasets.load_dataset(
    "shachardon/ShareLM", 
    split="train", 
    cache_dir=os.path.join(CACHE_ROOT, "datasets")
)

In [ ]:
parquet_path = os.path.join(CACHE_ROOT, "sharelm_full.parquet")
# ds.to_parquet(parquet_path
lazy_full_df = pl.scan_parquet(parquet_path)

In [ ]:
RESEARCH_DF_LEN = 100_000

In [ ]:
# sample 100

# lf = lazy_full_df.with_row_count("row_nr")
# total_rows = 4_500_000  # or compute lazily

# rand_idx = np.random.choice(total_rows, size=RESEARCH_DF_LEN, replace=False)

# df = (
#     lf.filter(pl.col("row_nr").is_in(rand_idx))
#     .collect(engine="streaming")
# )

In [ ]:
# df.write_parquet("research_df.pqt")
research_df = pl.read_parquet("research_df.pqt")
research_df.shape

In [ ]:
import polars as pl
import plotly.express as px

def plot_model_distribution(df: pl.DataFrame | pl.LazyFrame, n: int = 10):
    # 1. Normalize to LazyFrame (works for both eager & lazy)
    lf = df.lazy()

    # 2. Pure lazy pipeline
    counts_lf = (
        lf.group_by("model_name")
          .len()
          .rename({"len": "count"})
          .sort("count", descending=True)
    )

    # 3. Collect only once (boundary: plotting)
    counts = counts_lf.collect()

    # 4. Post-processing (small data now, safe in RAM)
    top_n = counts.head(n)
    other_val = counts.slice(n).select(pl.col("count").sum()).item()

    if other_val > 0:
        other_row = pl.DataFrame(
            {"model_name": ["Other"], "count": [other_val]},
            schema=top_n.schema
        )
        top_n = top_n.vstack(other_row)

    # 5. Plot
    fig = px.pie(
        top_n.to_pandas(),
        values="count",
        names="model_name",
        hole=0.3,
    )
    fig.update_layout(title=f"Top {n} Models")
    fig.show()


In [ ]:
plot_model_distribution(research_df)

In [ ]:
plot_model_distribution(lazy_full_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_top_users(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "user_id",
    n: int = 10,
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy aggregation (streaming-friendly)
    counts_lf = (
        lf.group_by(column_name)
          .len()
          .rename({"len": "count"})
          .sort("count", descending=True)
    )

    # 3. Collect once (boundary)
    counts = counts_lf.collect(streaming=True)

    # 4. Top-N + Other (small data now)
    top_n = counts.head(n)

    other_count = (
        counts
        .slice(n)
        .select(pl.col("count").sum())
        .item()
    )

    if other_count > 0:
        other_row = pl.DataFrame(
            {column_name: ["Other"], "count": [other_count]},
            schema=top_n.schema,
        )
        plot_df = top_n.vstack(other_row)
    else:
        plot_df = top_n

    # 5. Plot
    fig = px.pie(
        plot_df.to_pandas(),
        values="count",
        names=column_name,
        hole=0.3,
    )

    fig.update_layout(
        title={
            "text": f"Top {n} {column_name}s",
            "font": {"size": 14},
        }
    )

    fig.show()

In [ ]:
plot_top_users(research_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_events_by_year_month(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "timestamp",
    output_column: str = "YearMonth",
    date_format: str = "%Y-%m-%d %H:%M:%S.%f",
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy transformation + aggregation
    counts_lf = (
        lf.with_columns(
            pl.col(column_name)
            .str.replace("Z", "")
            .str.strptime(
                pl.Datetime,
                format=date_format,
                strict=False,
            )
            .dt.strftime("%Y-%m")
            .fill_null("Unknown")
            .alias(output_column)
        )
        .group_by(output_column)
        .len()
        .rename({"len": "Count"})
        .filter(pl.col(output_column) != "Unknown")
        .sort(output_column)
    )

    # 3. Collect once (streaming-safe)
    result = counts_lf.collect(streaming=True)

    # 4. Plot
    fig = px.bar(
        result.to_pandas(),
        x=output_column,
        y="Count",
        title="Count of Events by Year and Month",
        labels={output_column: "Month", "Count": "Event Count"},
    )

    fig.show()


In [ ]:
plot_events_by_year_month(research_df)

In [ ]:
import polars as pl
import plotly.express as px


def plot_conversation_length_distribution(
    df: pl.DataFrame | pl.LazyFrame,
    column_name: str = "conversation",
    x_max: int = 50,
):
    # 1. Normalize to LazyFrame
    lf = df.lazy()

    # 2. Lazy length computation + aggregation
    counts_lf = (
        lf.with_columns(
            pl.col(column_name)
            .list.len()
            .alias("conversation_len")
        )
        .group_by("conversation_len")
        .len()
        .rename({"len": "count"})
        .sort("conversation_len")
    )

    # 3. Collect once (streaming-safe)
    dist_df = counts_lf.collect(streaming=True)

    # Optional: clip to X_MAX for plotting
    dist_df = dist_df.filter(pl.col("conversation_len") <= x_max)

    # 4. Plot
    fig = px.bar(
        dist_df.to_pandas(),
        x="conversation_len",
        y="count",
        title="Distribution of Conversation Lengths",
        labels={
            "conversation_len": "Conversation Length (turns)",
            "count": "Frequency (Count)",
        },
    )

    fig.update_traces(opacity=0.8)

    fig.update_xaxes(
        range=[0, x_max],
        tickfont={"size": 8},
        title_font={"size": 12},
    )

    fig.update_layout(bargap=0.1)

    fig.show()


In [ ]:
plot_conversation_length_distribution(research_df)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

In [ ]:
BATCH_SIZE = 1000

def extract_prompts_batch(batch):
    return {
        "user_prompts": [
            [turn["content"] for turn in conversation if turn["role"] == "user"]
            for conversation in batch["conversation"]
        ]
    }

ours_dataset = ours_dataset.map(
    extract_prompts_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    num_proc=1,
    desc="Extracting User Prompts" 
)

In [ ]:
ours_dataset["user_prompts"][30_000]

In [ ]:
ours_dataset = ours_dataset.filter(lambda batch: [len(x) > 0 for x in batch["user_prompts"]], batched=True, desc="Filtering empty user prompt rows")
ours_dataset = ours_dataset.filter(lambda batch: [all([x is not None for x in user_prompts]) for user_prompts in batch["user_prompts"]], batched=True, desc="Filtering None prompts")
ours_dataset = ours_dataset.filter(lambda batch: [all([prompt != user_prompts[0] for prompt in user_prompts[1:]]) for user_prompts in batch["user_prompts"]], batched=True, desc="Filtering same prompts conversations")
ours_dataset.shape

In [ ]:
import string

def is_valid_prompt(prompt: str) -> bool:
    allowed_chars = set(string.printable)
    if not prompt.strip():
        return False
    is_printable_prompt = all(ch in allowed_chars for ch in prompt)
    return is_printable_prompt

In [ ]:
ours_dataset = ours_dataset.filter(lambda batch: [all([is_valid_prompt(prompt) for prompt in user_prompts]) for user_prompts in batch["user_prompts"]], batched=True, desc="Filtering invalid prompts")
ours_dataset.shape

In [ ]:
ours_dataset = ours_dataset.map(
    lambda batch: {"user_prompts_count": [len(prompts_list) for prompts_list in batch['user_prompts']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting User Prompts lengths" 
)

In [ ]:
# user prompts

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 2
MAX_THRESHOLD_CONVERSATION_LEN = 10
COLUMN_NAME = "user_prompts_count"

ours_dataset_medium_conversations = ours_dataset.filter(
    lambda batch: [
        (count >= MIN_THRESHOLD_CONVERSATION_LEN) and (count <= MAX_THRESHOLD_CONVERSATION_LEN)
        for count in batch[COLUMN_NAME]
    ],
    batched=True,
    desc="Filtering medium conversations"
)

ours_dataset_medium_conversations.shape

In [ ]:
BATCH_SIZE = 4

def extract_prompts_batch(batch):
    return {
        "model_answers": [
            [turn["content"] for turn in conversation if turn["role"] == "assistant"]
            for conversation in batch["conversation"]
        ]
    }

ours_dataset_medium_conversations = ours_dataset_medium_conversations.map(
    extract_prompts_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting Model answers" 
)

In [ ]:
ours_dataset_medium_conversations.shape

In [ ]:
#ours_dataset_medium_conversations.to_parquet("slurm/ours_dataset_medium_conversations.pqt")
#ours_dataset_medium_conversations = datasets.load_dataset("parquet", data_files="slurm/ours_dataset_medium_conversations.pqt")["train"]

In [ ]:
# extract vectors

In [ ]:
CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
ours_dataset_medium_conversations_with_cosines = datasets.load_dataset("parquet",
                                                                       data_files="slurm/ours_dataset_medium_conversations_with_cosines.pqt", 
                                                                       cache_dir=os.path.join(CACHE_ROOT, "datasets"))["train"]

In [ ]:
ours_dataset_medium_conversations_with_cosines.shape

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Sample 50 rows from the HF dataset
sampled = ours_dataset_medium_conversations_with_cosines.shuffle(seed=42).select(range(50))

# 2. Add a stable row_id BEFORE flattening/exploding
def add_row_id(example, idx):
    example["row_id"] = idx   # always exists, always correct
    return example

sampled = sampled.map(add_row_id, with_indices=True)

# 3. Add Rank = [1..len(similarity_to_first)]
def add_rank(example):
    n = len(example["user_prompts_similarity"])
    example["Rank"] = list(range(1, n + 1))
    return example

sampled = sampled.map(add_rank)

# 4. Flatten lists (HF explode-equivalent)
sampled = sampled.flatten_indices()

# 5. Convert to Pandas and explode list columns
df = sampled.to_pandas().explode(["user_prompts_similarity", "Rank"], ignore_index=True)

# 6. Parse numeric cosine similarity
df["cosine_value"] = pd.to_numeric(df["user_prompts_similarity"])

# 7. Plot
fig = px.line(
    df,
    x="Rank",
    y="cosine_value",
    color="row_id",
    line_group="row_id",
    markers=True,
    title="Cosine similarity scores from all to first user prompts",
    hover_data={"row_id": True, "Rank": True, "cosine_value": ':.3f'}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(
    xaxis_title="User prompt index",
    yaxis_title="Cosine similarity",
    legend_title="Sample Row",
    hovermode="closest"
)

fig.show()


In [ ]:
row_id = 18
print(sampled[row_id]["user_prompts_similarity"])
sampled[row_id]["user_prompts"]

In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"user_prompts_similarity_median": [np.median(similarity_to_first) for similarity_to_first in batch['user_prompts_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting user prompt similarity median" 
)

In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"user_prompts_similarity_mean": [np.mean(similarity_to_first) for similarity_to_first in batch['user_prompts_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting user prompt similarity median" 
)

In [ ]:
import plotly.express as px
import pandas as pd

# Extract the cosine_mean column ONLY (fast + memory safe)
cos_vals = ours_dataset_medium_conversations_with_cosines["user_prompts_similarity_mean"]

# Convert to a tiny pandas object for Plotly
df = pd.DataFrame({"cosine_mean": cos_vals})

# Bin settings (exact bin edges)
bin_settings = dict(
    start=-1,
    end=1,
    size=0.01
)

# Create histogram
fig = px.histogram(
    df,
    x='cosine_mean',
    nbins=10,  # this is ignored because xbins overrides bins
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity',
    labels={'cosine_mean': 'Mean Cosine Similarity Score', 'count': 'Frequency'},
    color_discrete_sequence=['#4C78A8']
)

# Apply explicit bin configuration
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black'))
)

# Layout styling
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Count)',
    bargap=0.005,
    template='plotly_white'
)

fig.show()


In [ ]:
SEMANTIC_CHANGE_COSINE_THRESHOLD = 0.85

def get_count_before_semantic_change(arr, threshold = SEMANTIC_CHANGE_COSINE_THRESHOLD):
    try:
        return next(i for i, x in enumerate(arr) if x < threshold)
    except StopIteration:
        
        return len(arr)

In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"count_before_prompt_semantic_change":
                   [get_count_before_semantic_change(similarity_to_first) for similarity_to_first in batch['user_prompts_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting count before prompt semantic change" 
)

In [ ]:
px.histogram(list(ours_dataset_medium_conversations_with_cosines["count_before_prompt_semantic_change"]))

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 3

number_of_high_semantic_conversations = ours_dataset_medium_conversations_with_cosines.filter(
    lambda x: x["count_before_prompt_semantic_change"] >= MIN_THRESHOLD_CONVERSATION_LEN
).num_rows

print(f"number of high semantic conversations: {number_of_high_semantic_conversations}")
print(f"all with bigger then {SEMANTIC_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
# model answers

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Sample 50 rows from the HF dataset
sampled = ours_dataset_medium_conversations_with_cosines.shuffle(seed=42).select(range(50))

# 2. Add a stable row_id BEFORE flattening/exploding
def add_row_id(example, idx):
    example["row_id"] = idx   # always exists, always correct
    return example

sampled = sampled.map(add_row_id, with_indices=True)

# 3. Add Rank = [1..len(similarity_to_first)]
def add_rank(example):
    n = len(example["model_answers_similarity"])
    example["Rank"] = list(range(1, n + 1))
    return example

sampled = sampled.map(add_rank)

# 4. Flatten lists (HF explode-equivalent)
sampled = sampled.flatten_indices()

# 5. Convert to Pandas and explode list columns
df = sampled.to_pandas().explode(["model_answers_similarity", "Rank"], ignore_index=True)

# 6. Parse numeric cosine similarity
df["cosine_value"] = pd.to_numeric(df["model_answers_similarity"])

# 7. Plot
fig = px.line(
    df,
    x="Rank",
    y="cosine_value",
    color="row_id",
    line_group="row_id",
    markers=True,
    title="Cosine similarity scores from all to first user prompts",
    hover_data={"row_id": True, "Rank": True, "cosine_value": ':.3f'}
)

fig.update_traces(mode="lines+markers")
fig.update_layout(
    xaxis_title="User prompt index",
    yaxis_title="Cosine similarity",
    legend_title="Sample Row",
    hovermode="closest"
)

fig.show()


In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"model_answers_similarity_median": [np.median(similarity_to_first) for similarity_to_first in batch['model_answers_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting model answers similarity median" 
)

In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"model_answers_similarity_mean": [np.mean(similarity_to_first) for similarity_to_first in batch['model_answers_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting model answers similarity mean" 
)

In [ ]:
import plotly.express as px
import pandas as pd

# Extract the cosine_mean column ONLY (fast + memory safe)
cos_vals = ours_dataset_medium_conversations_with_cosines["model_answers_similarity_mean"]

# Convert to a tiny pandas object for Plotly
df = pd.DataFrame({"cosine_mean": cos_vals})

# Bin settings (exact bin edges)
bin_settings = dict(
    start=-1,
    end=1,
    size=0.01
)

# Create histogram
fig = px.histogram(
    df,
    x='cosine_mean',
    nbins=10,  # this is ignored because xbins overrides bins
    histnorm='percent',
    title='Distribution of Mean Cosine Similarity',
    labels={'cosine_mean': 'Mean Cosine Similarity Score', 'count': 'Frequency'},
    color_discrete_sequence=['#4C78A8']
)

# Apply explicit bin configuration
fig.update_traces(
    xbins=bin_settings,
    marker=dict(line=dict(width=0.5, color='black'))
)

# Layout styling
fig.update_layout(
    xaxis_title='Mean Cosine Similarity (Bins of 0.01)',
    yaxis_title='Frequency (Count)',
    bargap=0.005,
    template='plotly_white'
)

fig.show()


In [ ]:
ours_dataset_medium_conversations_with_cosines = ours_dataset_medium_conversations_with_cosines.map(
    lambda batch: {"count_before_model_semantic_change":
                   [get_count_before_semantic_change(similarity_to_first) for similarity_to_first in batch['model_answers_similarity']]},
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Extracting count before model semantic change" 
)

In [ ]:
px.histogram(list(ours_dataset_medium_conversations_with_cosines["count_before_model_semantic_change"]))

In [ ]:
MIN_THRESHOLD_CONVERSATION_LEN = 3

total_high_semantic_conversations = ours_dataset_medium_conversations_with_cosines.filter(
    lambda x: x["count_before_model_semantic_change"] >= MIN_THRESHOLD_CONVERSATION_LEN)

print(f"number of high semantic conversations: {len(total_high_semantic_conversations)}")
print(f"all with bigger then {SEMANTIC_CHANGE_COSINE_THRESHOLD} cosine and more at least the {MIN_THRESHOLD_CONVERSATION_LEN} user turns")

In [ ]:
total_high_semantic_conversations.to_parquet("slurm/total_high_semantic_conversations.pqt")

In [ ]:
total_high_semantic_conversations_df = total_high_semantic_conversations.to_pandas()

In [ ]:
total_high_semantic_conversations_df.shape

In [ ]:
sample = total_high_semantic_conversations_df.sample()
sample["user_prompts"]

In [ ]:
l = sample.iloc[0]["user_prompts"]

In [ ]:
# use gpt3.5 to query about the prompts subject and so